In [25]:
import pandas as pd
from pathlib import Path
import os
import plotly.express as px
import geopandas as geopd
from data_preparation.utils.utils import get_s3, get_mapping

In [26]:
PATH_DATA = Path(os.getcwd()).parent / 'Output' / "data"
PATH_RAW_DATA = PATH_DATA / "raw_data"
PATH_PHEN = PATH_DATA / "final_data"

assert PATH_RAW_DATA.exists(), PATH_RAW_DATA
vodafone_file = PATH_RAW_DATA / "vodafone_attendences.csv"

PATH_PROCESSED = PATH_DATA / "data_processed"
assert PATH_PROCESSED.exists(), PATH_PROCESSED
phen_presenze = PATH_PHEN / "phen_presenze.parquet"

PATH_MAPPING = PATH_RAW_DATA / "mapping"
presenze_trentino_ispat = PATH_RAW_DATA / "presenze_Trentino_ISPAT.csv"
presenze_trentino_ispat_xalb = PATH_RAW_DATA / "presenze_Trentino_ISPAT_alb_xalb.csv"

In [27]:
def atp_to_comuni():
    id_to_apt = {}
    for apt, id_list in json_apt.items():
        for id_comune in id_list:
            # Trasformiamo in int e poi in stringa a 6 cifre: "22001" -> "022001"
            id_6_cifre = str(int(id_comune)).zfill(6)
            id_to_apt[id_6_cifre] = apt
    return id_to_apt

In [28]:
presenze = pd.read_parquet(phen_presenze)
vodafone_presenze = pd.read_csv(vodafone_file)
geojson_comuni_json_data = geopd.read_file(get_s3("TRENTINO-comuni_Vodafone_2023.geojson"))
json_vodafone = get_mapping("mapping_comuni_into_vodafone_Trento.json")
json_apt = get_mapping("map_comuni_into_apt.json")

In [29]:
location_map = dict(
    zip(
        geojson_comuni_json_data["id"].astype(str),
        geojson_comuni_json_data["name"].str.upper(),
    )
)

vodafone_presenze["comune"] = vodafone_presenze["locId"].astype(str).map(location_map)
vodafone_presenze["ID_COMUNE"] = vodafone_presenze["comune"].map(json_vodafone)

# Vigo di Fassa e Pozza di Fassa -> San Giovanni di Fassa
mask = vodafone_presenze["comune"].isin(["VIGO DI FASSA", "POZZA DI FASSA"])
vodafone_presenze.loc[mask, "comune"] = "SAN GIOVANNI DI FASSA"
vodafone_presenze.loc[mask, "ID_COMUNE"] = pd.Series(
    [[22250]] * mask.sum(), index=vodafone_presenze.index[mask]
)

vodafone_presenze_comuni = vodafone_presenze[
    vodafone_presenze["locType"] == "TN_MKT_AL_3"
]

In [30]:
# comuni Vodafone senza ID
print("locId senza ID_COMUNE:",
      vodafone_presenze_comuni[vodafone_presenze_comuni['ID_COMUNE'].isna()].locId.unique())
vodafone_presenze_comuni = vodafone_presenze_comuni.dropna(subset=['ID_COMUNE'])

locId senza ID_COMUNE: <ArrowStringArray>
[]
Length: 0, dtype: str


In [31]:
df_aggregato = (
    vodafone_presenze_comuni
    .groupby(['date', 'comune', 'userProfile'])
    .agg({'value': 'sum', 'ID_COMUNE': 'first'})
    .reset_index()
)

In [32]:
# correlations 
commuter = df_aggregato[df_aggregato['userProfile'] == 'COMMUTER']
corr_commuter = commuter.pivot(index='date', columns='comune', values='value').corr()

tourist = df_aggregato[df_aggregato['userProfile'] == 'TOURIST'].copy()
corr_tourist = tourist.pivot(index='date', columns='comune', values='value').corr()

In [33]:
# presenze_alb (pipeline) vs ISPAT per APT
presenze['DATA'] = pd.to_datetime(presenze['DATA'])
presenze['MESE'] = presenze['DATA'].dt.to_period("M")
df_grouped = presenze.groupby(["MESE", "ID_COMUNE"])['presenze_alb'].sum().reset_index()

df_grouped['ID_COMUNE'] = df_grouped['ID_COMUNE'].astype(str).str.zfill(6)
df_grouped['APT'] = df_grouped['ID_COMUNE'].map(atp_to_comuni())
print("Comuni senza APT:", df_grouped.loc[df_grouped['APT'].isna(), 'ID_COMUNE'].unique())
df_grouped.to_csv("grouped.csv", index=False)

Comuni senza APT: <ArrowStringArray>
[]
Length: 0, dtype: str


In [34]:
df_grouped['Anno'] = df_grouped['MESE'].dt.year
df_grouped['Mese'] = df_grouped['MESE'].dt.month
df_grouped_pres_summed = (
    df_grouped.groupby(["APT", "Anno", "Mese"])['presenze_alb'].sum()
    .reset_index().rename(columns={'APT': 'Ambito'})
)

pres_off_df_apt = pd.read_csv(presenze_trentino_ispat).sort_values("Ambito")
mg = pres_off_df_apt.merge(df_grouped_pres_summed, on=['Anno', 'Mese', 'Ambito'])
print(f"Righe ISPAT: {len(pres_off_df_apt)} | righe con corrispondenza: {len(mg)}")
mg['diff'] = mg['Presenze'] - mg['presenze_alb']
print(mg['diff'].describe())
mg[abs(mg['diff']) > 15]


Righe ISPAT: 468 | righe con corrispondenza: 324
count      324.000000
mean       461.438272
std       2062.693937
min          0.000000
25%          0.000000
50%          0.000000
75%         48.500000
max      24645.000000
Name: diff, dtype: float64


,Ambito,Anno,Mese,Presenze,presenze_alb,diff
0,Altipiani cimbri e Vigolana,2022,1,70025,68896.0,1129.0
1,Altipiani cimbri e Vigolana,2022,2,69073,67839.0,1234.0
16,Altipiani cimbri e Vigolana,2023,5,2590,2401.0,189.0
21,Altipiani cimbri e Vigolana,2023,10,3695,3575.0,120.0
22,Altipiani cimbri e Vigolana,2023,11,1285,1219.0,66.0
...,...,...,...,...,...,...
311,"Valsugana, Tesino e Valle dei Mocheni",2023,3,11427,11379.0,48.0
316,"Valsugana, Tesino e Valle dei Mocheni",2023,8,95832,95725.0,107.0
321,"Valsugana, Tesino e Valle dei Mocheni",2024,1,15657,15555.0,102.0
322,"Valsugana, Tesino e Valle dei Mocheni",2024,2,11838,11740.0,98.0


In [35]:
pres_off_df_apt_noprov = pres_off_df_apt[pres_off_df_apt['Ambito'] != 'Provincia'].copy()
summed = pres_off_df_apt_noprov.groupby(["Anno", "Mese"])['Presenze'].sum().reset_index()

pres_off_xalb = pd.read_csv(presenze_trentino_ispat_xalb)
mgd = pres_off_xalb.merge(summed, on=['Anno', 'Mese'])
mgd['diff'] = mgd['Presenze alberghi'] - mgd['Presenze']
print(mgd['diff'].describe())


count    36.0
mean      0.0
std       0.0
min       0.0
25%       0.0
50%       0.0
75%       0.0
max       0.0
Name: diff, dtype: float64


In [36]:
# split tourists equally
tourist['date'] = pd.to_datetime(tourist['date'])
tourist['mese'] = tourist['date'].dt.to_period('M')

tourist_ids = tourist[['date', 'mese', 'comune', 'ID_COMUNE', 'value']].copy()
tourist_ids['num_ids'] = tourist_ids['ID_COMUNE'].apply(lambda x: len(x) if isinstance(x, list) else 1)
tourist_ids['value'] = tourist_ids['value'] / tourist_ids['num_ids']

tourist_ids = tourist_ids.explode('ID_COMUNE').dropna(subset=['ID_COMUNE'])
tourist_ids['ID_NORM'] = tourist_ids['ID_COMUNE'].astype(str).str.strip().str.lstrip('0')

tourist_month = (
    tourist_ids.groupby(['mese', 'ID_NORM'], as_index=False)['value']
    .sum()
    .rename(columns={'value': 'presenze_vodafone'})
)

In [37]:
#  ISPAT per APT
pres_off_df_apt_noprov['mese'] = pd.to_datetime(
    pres_off_df_apt_noprov['Anno'].astype(str) + '-' + pres_off_df_apt_noprov['Mese'].astype(str).str.zfill(2)
).dt.to_period('M')
pres_off_df_apt_noprov['ID'] = pres_off_df_apt_noprov['Ambito'].map(json_apt)
print("Ambiti senza lista comuni:", pres_off_df_apt_noprov.loc[pres_off_df_apt_noprov['ID'].isna(), 'Ambito'].unique())

df_esploso_apt_no_prov = pres_off_df_apt_noprov[['Ambito', 'mese', 'ID']].explode('ID')
df_esploso_apt_no_prov = df_esploso_apt_no_prov.dropna(subset=['ID']).copy()
df_esploso_apt_no_prov['ID_NORM'] = df_esploso_apt_no_prov['ID'].astype(str).str.strip().str.lstrip('0')

# checks 
print("Comuni degli ambiti senza Vodafone:",
      sorted(set(df_esploso_apt_no_prov['ID_NORM']) - set(tourist_month['ID_NORM'])))
print("Comuni Vodafone senza ambito:",
      sorted(set(tourist_month['ID_NORM']) - set(df_esploso_apt_no_prov['ID_NORM'])))
print("Comuni presenti in piu' ambiti nello stesso mese:",
      df_esploso_apt_no_prov.duplicated(['mese', 'ID_NORM']).sum())


Ambiti senza lista comuni: <ArrowStringArray>
[]
Length: 0, dtype: str
Comuni degli ambiti senza Vodafone: []
Comuni Vodafone senza ambito: []
Comuni presenti in piu' ambiti nello stesso mese: 0


In [45]:
#  Vodafone per ambito e mese
merged_apt_voda = df_esploso_apt_no_prov.merge(tourist_month, on=['mese', 'ID_NORM'], how='left')
df_grouped_apt = merged_apt_voda.groupby(['Ambito', 'mese'])['presenze_vodafone'].sum().reset_index()
cm = merged_apt_voda[merged_apt_voda['mese'] < '2024-04']
cm[cm.presenze_vodafone.isna()]  # 22078 DRENA, 22191 TENNO, 22018 BOCEGNAGO, 22164 SAGRON-MIS, 22059 CINTE TESINO


,Ambito,mese,ID,ID_NORM,presenze_vodafone
609,"Garda trentino, Valle di Ledro, Terme di Coman...",2022-03,22078,22078,NaN
853,"Garda trentino, Valle di Ledro, Terme di Coman...",2023-07,22191,22191,NaN
1232,"Madonna di Campiglio, Pinzolo, Val Rendena, Gi...",2022-07,22018,22018,NaN
1251,"Madonna di Campiglio, Pinzolo, Val Rendena, Gi...",2022-08,22018,22018,NaN
1270,"Madonna di Campiglio, Pinzolo, Val Rendena, Gi...",2022-09,22018,22018,NaN
2415,"San Martino di Castrozza, Primiero e Vanoi",2022-01,22164,22164,NaN
4938,"Valsugana, Tesino e Valle dei Mocheni",2022-01,22059,22059,NaN


In [58]:
tourist_ids[(tourist_ids['ID_COMUNE'] == 22078) & (tourist_ids['mese'] == '2022-03')]
tourist_ids[(tourist_ids['ID_COMUNE'] == 22059) & (tourist_ids['mese'] == '2022-01')]

,date,mese,comune,ID_COMUNE,value,num_ids,ID_NORM


In [39]:
mgdt = pres_off_df_apt_noprov.merge(df_grouped_apt, on=['mese', 'Ambito'], how='outer', indicator=True)
print(mgdt['_merge'].value_counts())
mgdt = mgdt[mgdt['_merge'] == 'both'].drop(columns='_merge')
mgdt = mgdt[mgdt['presenze_vodafone'] > 0].copy()  # evita divisioni per zero / mesi senza dato Vodafone
mgdt['ratio'] = mgdt['Presenze'] / mgdt['presenze_vodafone']


_merge
both          432
left_only       0
right_only      0
Name: count, dtype: int64


In [ ]:
PROV = (
    mgdt.groupby('mese')
    .agg(presenze_ispat=('Presenze', 'sum'), presenze_vodafone=('presenze_vodafone', 'sum'))
    .reset_index()
)
PROV['ratio'] = PROV['presenze_ispat'] / PROV['presenze_vodafone']
APT = mgdt[['Ambito', 'mese', 'ratio']]

,Ambito,mese,ratio_ambito,ratio_prov,scarto_assoluto,scarto_relativo,mese_str
127,Val di Fassa,2022-11,0.208226,0.076148,0.132077,1.734468,2022-11
67,Val di Fassa,2022-06,0.518392,0.244242,0.274150,1.122452,2022-06
271,Val di Fassa,2023-11,0.311221,0.147776,0.163444,1.106027,2023-11
46,Val di Sole,2022-04,0.322585,0.154896,0.167689,1.082590,2022-04
122,"Garda trentino, Valle di Ledro, Terme di Coman...",2022-11,0.158256,0.076148,0.082107,1.078250,2022-11
...,...,...,...,...,...,...,...
267,"Madonna di Campiglio, Pinzolo, Val Rendena, Gi...",2023-11,0.149020,0.147776,0.001244,0.008417,2023-11
206,"Garda trentino, Valle di Ledro, Terme di Coman...",2023-06,0.374703,0.376799,-0.002096,-0.005562,2023-06
291,"Madonna di Campiglio, Pinzolo, Val Rendena, Gi...",2024-01,0.426180,0.428081,-0.001901,-0.004441,2024-01
281,"San Martino di Castrozza, Primiero e Vanoi",2023-12,0.320893,0.320384,0.000509,0.001588,2023-12


In [41]:
# APT vs provincia 
confronto = APT.merge(PROV[['mese', 'ratio']], on='mese', suffixes=('_ambito', '_prov'))
confronto['scarto_assoluto'] = confronto['ratio_ambito'] - confronto['ratio_prov']
confronto['scarto_relativo'] = confronto['scarto_assoluto'] / confronto['ratio_prov']
confronto = confronto.sort_values('scarto_relativo', key=abs, ascending=False)
confronto['mese_str'] = confronto['mese'].astype(str)

#  Grafico
fig = px.line(
    confronto.sort_values('mese'),
    x='mese_str', y='ratio_ambito', color='Ambito',
    title='Rapporto Presenze alberghiere ISPAT / Vodafone turisti, per ambito nel tempo'
)
prov_ref = PROV.sort_values('mese')
fig.add_scatter(
    x=prov_ref['mese'].astype(str), y=prov_ref['ratio'],
    mode='lines', name='Rapporto provinciale (somma ambiti)',
    line=dict(color='black', dash='dash')
)
fig.show()